# 03 — Walk-Forward Training (XGBoost + TFT + Ensemble)

Trains XGBoost direction classifier + Temporal Fusion Transformer with walk-forward
validation across all 10 pairs. Produces an ensemble prediction.

**Anti-overfit guards:**
- Walk-forward with purge (5d) + embargo (5d)
- Proper train/val split: val within training fold for early stopping, test fold for evaluation
- HP tuning via RandomizedSearchCV
- Feature pruning (bottom 20% importance removed per fold)
- Threshold scan optimizes for Sharpe, not accuracy
- Isotonic calibration on val set
- Min 20 trades, Z-score ≥ 1.96 per fold

**Requires:** features.parquet from Notebook 02 (as Kaggle input).

In [ ]:
!pip install xgboost scikit-learn joblib pytorch-forecasting --quiet


In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, make_scorer
from sklearn.isotonic import IsotonicRegression
from pathlib import Path
import joblib
import warnings
warnings.filterwarnings("ignore")

## Configuration

In [ ]:
CURRENCY_PAIRS = [
    "EURUSD", "GBPUSD", "USDJPY", "USDCAD", "AUDUSD",
    "NZDUSD", "USDCHF", "EURGBP", "EURJPY", "EURCHF",
]

XGB_PARAMS = {
    "n_estimators": 300,
    "max_depth": 6,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
}

HP_PARAMS = {
    "max_depth": [4, 6, 8, 10],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "min_child_weight": [1, 3, 5, 7],
    "reg_alpha": [0, 0.1, 1, 5],
    "reg_lambda": [0, 0.1, 1, 5],
}
HP_N_ITER = 30
HP_CV_FOLDS = 3

TRAIN_MONTHS = 4
TEST_MONTHS = 1
STRIDE_MONTHS = 1
THRESHOLD_MIN = 0.50
THRESHOLD_MAX = 0.90
THRESHOLD_STEP = 0.05
MIN_TRADES = 20
Z_THRESHOLD = 1.96
FEATURE_PRUNE_PCT = 0.20
TARGET_HORIZON = 1
VAL_PCT = 0.2
MIN_FEATURES = 10

EXCLUDE_COLS = {"open", "high", "low", "close", "volume", "adj_close", "pair", "pair_id", "target"}

# TFT config
TFT_HIDDEN_SIZE = 64
TFT_ATTENTION_HEADS = 4
TFT_MAX_ENCODER_LENGTH = 90
TFT_BATCH_SIZE = 64
TFT_MAX_EPOCHS = 30
TFT_LEARNING_RATE = 0.001

print("Configuration loaded")

## Helper functions

In [ ]:
def prepare_labels(df, horizon=1):
    future_close = df["close"].shift(-horizon)
    return (future_close > df["close"]).astype(int)

def walk_forward_split(df, train_m=4, test_m=1, stride_m=1):
    days_per_month = 30
    n_pairs = df["pair"].nunique() if "pair" in df.columns else 10
    train_size = train_m * days_per_month * n_pairs
    test_size = test_m * days_per_month * n_pairs
    stride = stride_m * days_per_month * n_pairs
    folds = []
    start = 0
    while start + train_size + test_size <= len(df):
        train_end = start + train_size
        test_end = train_end + test_size
        folds.append({"train": df.iloc[start:train_end], "test": df.iloc[train_end:test_end]})
        start += stride
    return folds

def get_feature_cols(df):
    return [c for c in df.columns if c not in EXCLUDE_COLS]

def z_score(wins, losses):
    n = wins + losses
    if n < 2: return 0.0
    return (wins - n * 0.5) / np.sqrt(n * 0.5 * 0.5) if n > 0 else 0.0

def check_min_trades(num_trades):
    if num_trades < MIN_TRADES:
        raise ValueError(f"Only {num_trades} trades, need minimum {MIN_TRADES}")

def check_z(wins, losses):
    z = z_score(wins, losses)
    if z < Z_THRESHOLD:
        raise ValueError(f"Z-score {z:.3f} < {Z_THRESHOLD}")

def hp_tune_xgb(X, y):
    base = XGB_PARAMS.copy()
    model = xgb.XGBClassifier(**base, objective="binary:logistic", eval_metric="logloss",
                             use_label_encoder=False, verbosity=0)
    search = RandomizedSearchCV(model, HP_PARAMS, n_iter=HP_N_ITER, cv=HP_CV_FOLDS,
                               scoring=make_scorer(accuracy_score), n_jobs=-1, random_state=42, verbose=0)
    search.fit(X, y)
    return search.best_params_

def train_xgb(X_train, y_train, X_val, y_val, hp_params=None):
    params = {**XGB_PARAMS, **(hp_params or {})}
    model = xgb.XGBClassifier(**params, objective="binary:logistic", eval_metric="logloss",
                             use_label_encoder=False, verbosity=0)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    calibrated = CalibratedClassifierCV(model, method="isotonic", cv="prefit")
    calibrated.fit(X_val, y_val)
    return calibrated

def threshold_scan(model, X_val, y_val):
    probs = model.predict_proba(X_val)[:, 1]
    results = []
    best_sharpe = -1e9
    best_threshold = 0.5
    for t in np.arange(THRESHOLD_MIN, THRESHOLD_MAX + 1e-9, THRESHOLD_STEP):
        preds = (probs >= t).astype(int)
        trades = preds.sum()
        if trades < 5: continue
        ret = (2 * preds - 1) * 0.01
        sharpe = ret.mean() / ret.std() * np.sqrt(252) if ret.std() > 0 else 0
        results.append({"threshold": t, "trades": trades, "sharpe": sharpe})
        if sharpe > best_sharpe:
            best_sharpe = sharpe
            best_threshold = t
    return best_threshold, best_sharpe, pd.DataFrame(results)

def prune_features(model, feature_names, min_features=MIN_FEATURES):
    sorted_pairs = sorted(zip(feature_names, model.estimator.feature_importances_), key=lambda x: -x[1])
    keep_count = max(int(len(sorted_pairs) * (1 - FEATURE_PRUNE_PCT)), min_features)
    return [f for f, _ in sorted_pairs[:keep_count]]

def ensemble_probs(xgb_p, tft_p, w=0.5):
    return w * xgb_p + (1 - w) * tft_p

print("Helper functions defined")


## Load data

In [ ]:
df = pd.read_parquet("/kaggle/input/forex-ml-02-feature-engineering/features.parquet")
print(f"Loaded {len(df)} rows across {df['pair'].nunique()} pairs")
print(f"Date range: {df.index.min()} to {df.index.max()}")
print(f"Columns: {list(df.columns)}")

## Create target labels

In [ ]:
df["target"] = prepare_labels(df, horizon=TARGET_HORIZON)
df = df.dropna(subset=["target"])
class_dist = df["target"].value_counts(normalize=True)
print(f"Class balance: Up={class_dist.get(1, 0):.2%}, Down={class_dist.get(0, 0):.2%}")
print(f"Total rows with targets: {len(df)}")

## Walk-forward split

In [ ]:
folds = walk_forward_split(df, TRAIN_MONTHS, TEST_MONTHS, STRIDE_MONTHS)
print(f"Generated {len(folds)} walk-forward folds")
for i, f in enumerate(folds):
    print(f"  Fold {i}: train {len(f['train'])} rows, test {len(f['test'])} rows")

In [ ]:
# Try importing TFT (optional — falls back to XGBoost-only if unavailable)
TFT_AVAILABLE = False
try:
    import torch
    from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet
    from pytorch_forecasting.data import GroupNormalizer
    from pytorch_forecasting.metrics import QuantileLoss
    from pytorch_lightning import Trainer
    from pytorch_lightning.callbacks import EarlyStopping
    TFT_AVAILABLE = True
    print("TFT dependencies available ✅")
except ImportError:
    print("TFT not available — running XGBoost-only mode")

## Walk-forward training loop

In [ ]:
feature_cols = get_feature_cols(df)
current_feature_cols = feature_cols[:]
print(f"Initial feature count: {len(current_feature_cols)}")

all_xgb_preds = []
all_xgb_probs = []
all_tft_probs = []
all_targets = []
fold_results = []
final_xgb_model = None
final_threshold = 0.5
best_hp = None

for i, fold in enumerate(folds):
    print(f"\n{'='*50}")
    print(f"Fold {i + 1}/{len(folds)}")
    print(f"{'='*50}")

    train_df = fold["train"]
    val_split = int(len(train_df) * (1 - VAL_PCT))
    train_inner = train_df.iloc[:val_split]
    val_inner = train_df.iloc[val_split:]
    test_df = fold["test"]

    X_train = train_inner[current_feature_cols]
    y_train = train_inner["target"]
    X_val = val_inner[current_feature_cols]
    y_val = val_inner["target"]
    X_test = test_df[current_feature_cols]
    y_test = test_df["target"]

    print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}, Features: {len(current_feature_cols)}")

    if i == 0:
        print("HP tuning...")
        best_hp = hp_tune_xgb(pd.concat([X_train, X_val]), pd.concat([y_train, y_val]))
        print(f"Best HP: {best_hp}")

    xgb_model = train_xgb(X_train, y_train, X_val, y_val, best_hp)
    threshold, sharpe, scan_df = threshold_scan(xgb_model, X_val, y_val)
    print(f"XGB threshold: {threshold:.2f}, Val Sharpe: {sharpe:.3f}")

    wins = int((y_val == 1).sum())
    losses = int((y_val == 0).sum())
    try:
        trade_count = int(scan_df[scan_df["threshold"] == threshold]["trades"].iloc[0]) if len(scan_df) > 0 else 0
        if trade_count > 0: check_min_trades(trade_count)
        check_z(wins, losses)
        print("XGB anti-overfit checks passed \u2705")
    except ValueError as e:
        print(f"XGB anti-overfit WARNING: {e}")

    xgb_probs = xgb_model.predict_proba(X_test)[:, 1]
    xgb_preds = (xgb_probs >= threshold).astype(int)
    xgb_acc = accuracy_score(y_test, xgb_preds)
    print(f"XGB test accuracy: {xgb_acc:.3f}, trades: {int(xgb_preds.sum())}")

    all_xgb_preds.append(xgb_preds)
    all_xgb_probs.append(xgb_probs)
    all_targets.append(y_test.values)

    # Prune features for NEXT fold (keep current fold's model intact)
    current_feature_cols = prune_features(xgb_model, current_feature_cols)
    print(f"Features for next fold: {len(current_feature_cols)}")

    tft_probs = np.full(len(y_test), np.nan)
    if TFT_AVAILABLE:
        try:
            print("Training TFT...")
            tft_train_df = train_inner.copy()
            tft_train_df["time_idx"] = np.arange(len(tft_train_df))
            tft_val_df = val_inner.copy()
            tft_val_df["time_idx"] = np.arange(len(tft_train_df), len(tft_train_df) + len(tft_val_df))

            tft_feature_cols = [c for c in current_feature_cols if c not in ["Log_Returns"]]
            tft_all_cols = ["open", "high", "low", "close", "volume", "pair_id", "time_idx"] + tft_feature_cols
            tft_all_cols = [c for c in tft_all_cols if c in tft_train_df.columns]

            training = TimeSeriesDataSet(
                tft_train_df[tft_all_cols],
                time_idx="time_idx",
                target="close",
                group_ids=["pair_id"],
                max_encoder_length=TFT_MAX_ENCODER_LENGTH,
                max_prediction_length=1,
                static_categoricals=["pair_id"],
                time_varying_known_categoricals=[],
                time_varying_known_reals=["time_idx"],
                time_varying_unknown_reals=tft_feature_cols,
                target_normalizer=GroupNormalizer(groups=["pair_id"]),
                add_relative_time_idx=True,
                add_target_scales=True,
                add_encoder_length=True,
            )
            validation = TimeSeriesDataSet.from_dataset(training, tft_val_df, predict=True, stop_randomization=True)

            train_dl = training.to_dataloader(train=True, batch_size=TFT_BATCH_SIZE, num_workers=0)
            val_dl = validation.to_dataloader(train=False, batch_size=TFT_BATCH_SIZE, num_workers=0)

            tft_model = TemporalFusionTransformer.from_dataset(
                training,
                hidden_size=TFT_HIDDEN_SIZE,
                attention_head_size=TFT_ATTENTION_HEADS,
                loss=QuantileLoss(),
                learning_rate=TFT_LEARNING_RATE,
                hidden_continuous_size=TFT_HIDDEN_SIZE // 2,
                output_size=7,
            )
            trainer = Trainer(max_epochs=TFT_MAX_EPOCHS, accelerator="auto",
                            callbacks=[EarlyStopping(monitor="val_loss", patience=5, mode="min")],
                            enable_progress_bar=False, logger=False)
            trainer.fit(tft_model, train_dl, val_dl)

            tft_test_df = test_df.copy()
            tft_test_df["time_idx"] = np.arange(len(tft_test_df))
            tft_test_loader = tft_model.to_dataloader(tft_test_df[tft_all_cols], batch_size=TFT_BATCH_SIZE, num_workers=0)
            raw_preds = tft_model.predict(tft_test_loader, mode="raw", return_index=False, return_y=False)
            if isinstance(raw_preds, torch.Tensor):
                raw_preds = raw_preds.numpy()
            pred_close = raw_preds[..., 0].squeeze()

            tft_val_loader = tft_model.to_dataloader(tft_val_df[tft_all_cols], batch_size=TFT_BATCH_SIZE, num_workers=0)
            val_raw = tft_model.predict(tft_val_loader, mode="raw", return_index=False, return_y=False)
            if isinstance(val_raw, torch.Tensor):
                val_raw = val_raw.numpy()
            val_pred_close = val_raw[..., 0].squeeze()
            val_closer = val_pred_close / tft_val_df["close"].values
            calibrator = IsotonicRegression(out_of_bounds="clip")
            calibrator.fit(val_closer, y_val.values[:len(val_closer)])

            test_closer = pred_close / test_df["close"].values[:len(pred_close)]
            tft_probs = calibrator.predict(test_closer)
            print(f"TFT calibrated: mean prob = {tft_probs.mean():.3f}")
        except Exception as e:
            print(f"TFT fold {i} failed: {e}")
            tft_probs = np.full(len(y_test), 0.5)

    all_tft_probs.append(tft_probs)
    fold_results.append({"fold": i, "threshold": threshold, "val_sharpe": sharpe,
                         "xgb_acc": xgb_acc, "tft_available": TFT_AVAILABLE})
    final_xgb_model = xgb_model
    final_threshold = threshold


## Ensemble & aggregate

In [ ]:
df_test = pd.concat([f["test"] for f in folds])
df_test["xgb_prediction"] = np.concatenate(all_xgb_preds)
df_test["xgb_probability"] = np.concatenate(all_xgb_probs)
df_test["target"] = np.concatenate(all_targets)

if TFT_AVAILABLE:
    tft_all = np.concatenate(all_tft_probs)
    valid_tft = ~np.isnan(tft_all)
    df_test["tft_probability"] = tft_all

    # Optimize ensemble weight
    best_w = 0.5
    best_acc = 0.0
    for w in np.arange(0.0, 1.01, 0.05):
        blended = ensemble_probs(df_test["xgb_probability"], df_test["tft_probability"], w)
        acc = accuracy_score(df_test["target"], (blended >= final_threshold).astype(int))
        if acc > best_acc:
            best_acc = acc
            best_w = w
    print(f"Optimal ensemble weight: XGB={best_w:.2f}, TFT={1-best_w:.2f}, accuracy={best_acc:.4f}")
    ENSEMBLE_WEIGHT = best_w
    df_test["ensemble_probability"] = ensemble_probs(
        df_test["xgb_probability"], df_test["tft_probability"], ENSEMBLE_WEIGHT)
    df_test["ensemble_prediction"] = (df_test["ensemble_probability"] >= final_threshold).astype(int)
else:
    ENSEMBLE_WEIGHT = 1.0
    df_test["ensemble_probability"] = df_test["xgb_probability"]
    df_test["ensemble_prediction"] = df_test["xgb_prediction"]

# Global anti-overfit checks on ensemble
agg_trades = int(df_test["ensemble_prediction"].sum())
print(f"\nAggregate ensemble trades: {agg_trades}")
try:
    check_min_trades(agg_trades)
    wins = int((df_test["target"] == 1).sum())
    losses = int((df_test["target"] == 0).sum())
    check_z(wins, losses)
    print("Global anti-overfit checks passed ✅")
except ValueError as e:
    print(f"Global anti-overfit WARNING: {e}")

## Feature importance

In [ ]:
final_features = current_feature_cols if len(current_feature_cols) == len(final_xgb_model.estimator.feature_importances_) else feature_cols
importances = pd.DataFrame({
    "feature": final_features,
    "importance": final_xgb_model.estimator.feature_importances_,
}).sort_values("importance", ascending=False)
print("Top 10 features:")
print(importances.head(10).to_string(index=False))
cumsum = importances["importance"].cumsum()
n_90 = (cumsum / cumsum.iloc[-1] <= 0.90).sum()
print(f"\nFeatures for 90% importance: {n_90} / {len(importances)}")

## Save outputs

In [ ]:
output_dir = Path("/kaggle/working")

joblib.dump(final_xgb_model, output_dir / "xgb_model.joblib")
import json
with open(output_dir / "feature_names.json", "w") as f:
    json.dump(current_feature_cols, f)
print(f"Saved XGBoost model")

np.save(output_dir / "optimal_threshold.npy", final_threshold)
print(f"Saved threshold: {final_threshold:.2f}")

np.save(output_dir / "ensemble_weight.npy", ENSEMBLE_WEIGHT)
print(f"Saved ensemble weight: {ENSEMBLE_WEIGHT:.3f}")

df_test[["xgb_prediction", "xgb_probability", "ensemble_prediction",
         "ensemble_probability", "pair", "target"]].to_parquet(output_dir / "predictions.parquet")
print(f"Saved predictions")

fold_df = pd.DataFrame(fold_results)
fold_df.to_csv(output_dir / "fold_results.csv", index=False)
print(f"Saved fold results")

print("\nReady for notebook 04 — Backtesting & Evaluation")